# 05 — Transfer Learning ACC Arena → Salt&Tar (Advanced, 3 pts)
Reuse the NN trained on **ACC Arena** and **fine-tune** it on a *limited* **Salt&Tar** training set, then
compare against the **same network trained from scratch** on that limited set.

We reuse the exact feature schema from notebook 02 (fixed numeric + 6 traffic one-hot + `BEST_X` neighbours)
so the ACC-trained weights transfer directly.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 60            # downsampling step in seconds. Lower = more samples, more time (try 30 or 15).
N_USERS          = 1500          # ACC Arena users to load. Closest-users neighbours are searched WITHIN this subset.
N_USERS_SALT     = 1000          # Salt&Tar users (transfer-learning notebook)
X_VALUES         = [1, 3, 5, 10] # number of closest users to experiment with
BEST_X           = 5             # X used for the transfer-learning experiment (pick the best from notebook 04)
LOG_TARGET       = True          # train on log1p(throughput) (target is very skewed); metrics stay in Mbps
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def metric_files(venue_dir, subdir_glob, file_glob, n_files=None):
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)),
                   key=lambda p: int(re.search(r'_(\d+)_\d+\.csv$', p).group(1)))
    return files[:n_files] if n_files else files

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        df = pd.read_csv(f).rename(columns={pd.read_csv(f, nrows=0).columns[0]: "time"}).set_index("time")
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in df.columns}
        df = df[[c for c, u in cmap.items() if u in user_ids]].rename(columns=cmap)
        df = _align(df, grid, tol)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState).
    lat/lon are converted to local metres; mobileState is the traffic_type."""
    frames = []
    for f in files:
        df = pd.read_csv(f)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        lat, lon, alt, mob = arr[:, 2::5], arr[:, 3::5], arr[:, 4::5], arr[:, 5::5]
        lat0, lon0 = np.nanmean(lat), np.nanmean(lon)
        R = 6371000.0
        x = R * np.radians(lon - lon0) * math.cos(math.radians(lat0))
        y = R * np.radians(lat - lat0)
        def mk(v, name):
            d = pd.DataFrame(v, index=t, columns=ids)
            d = d[[u for u in ids if u in user_ids]]
            d.index.name = "time"
            return _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
        m = mk(x, "x").merge(mk(y, "y"), on=["time", "user_id"]) \
                      .merge(mk(alt, "z"), on=["time", "user_id"]) \
                      .merge(mk(mob, "traffic_type"), on=["time", "user_id"])
        frames.append(m)
    return pd.concat(frames, ignore_index=True)

def assemble(venue_key, n_users, resample_seconds):
    """Return a tidy DataFrame with one row per (user, timestamp)."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    user_ids = set(range(n_users))
    n_files = math.ceil(n_users / 500)
    tol = resample_seconds / 2
    ref = metric_files(venue, "*Throughput*", "*.csv", 1)[0]
    grid = build_grid(ref, resample_seconds)
    mf = lambda sub, fg: metric_files(venue, sub, fg, n_files)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

In [ ]:
# === Closest-users feature engineering (Team-8 specific) ===
# For each (timestamp, user) we append the features of its X nearest users at that
# instant (3-D Euclidean distance on x,y,z). Per neighbour: [dist, sinr_dl, sinr_ul, prb, bler, throughput].
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler", "throughput"]

def add_closest_user_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan)
    feat = df[NEIGHBOR_FEATS].values
    for _, idx in df.groupby("time").groups.items():
        idx = np.asarray(idx)
        if len(idx) <= 1:
            continue
        tree = cKDTree(df.loc[idx, ["x", "y", "z"]].values)
        k = min(x_max + 1, len(idx))
        dist, nb = tree.query(df.loc[idx, ["x", "y", "z"]].values, k=k)
        for r in range(len(idx)):
            picks = [j for j in range(k) if nb[r, j] != r][:x_max]
            vals = []
            for j in picks:
                vals.append(dist[r, j]); vals.extend(feat[idx[nb[r, j]]])
            out[idx[r], :len(vals)] = vals
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

def neighbor_cols(x):
    cols = []
    for k in range(x):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    return cols

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

## Build the Salt&Tar dataset (limited) with the same feature schema

In [ ]:
import json, joblib, numpy as np, pandas as pd
df = assemble("salt", n_users=N_USERS_SALT, resample_seconds=RESAMPLE_SECONDS)
df = add_closest_user_features(df, x_max=BEST_X)
if ACTIVE_ONLY:                                  # same target definition as ACC Arena (notebook 02)
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
print("Salt&Tar tidy:", df.shape, "| traffic types:", sorted(df.traffic_type.unique()))

BASE_NUM = ["bler","prb","sinr_dl","sinr_ul","x","y","z"]
TRAFFIC_CLASSES = [0,1,2,3,4,5]
acc_cols = json.load(open(f"{PROCESSED_DIR}/acc_X{BEST_X}_cols.json"))
scaler   = joblib.load(f"{PROCESSED_DIR}/acc_X{BEST_X}_scaler.pkl")

def build_like_acc(frame):
    num_cols = BASE_NUM + neighbor_cols(BEST_X)
    num = frame[num_cols].astype(float); num = num.fillna(num.median())
    num_s = pd.DataFrame(scaler.transform(num), columns=num_cols, index=frame.index)
    oh = pd.DataFrame({f"traffic_{c}": (frame.traffic_type==c).astype(float) for c in TRAFFIC_CLASSES},
                      index=frame.index)
    X = pd.concat([num_s, oh], axis=1).reindex(columns=acc_cols, fill_value=0.0)
    return X.values.astype("float32")

## Limited Salt&Tar split (small train, fair test)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n_train_users = max(20, int(0.15*len(users)))   # "limited" target-domain training set
tr_u = set(users[:n_train_users]); te_u = set(users[n_train_users:])
tr = df[df.user_id.isin(tr_u)]; te = df[df.user_id.isin(te_u)]
Xtr, ytr = build_like_acc(tr), tr.throughput.values.astype("float32")
Xte, yte = build_like_acc(te), te.throughput.values.astype("float32")
print("train rows:", len(Xtr), "| test rows:", len(Xte))

## (a) Fine-tune the ACC-trained NN  vs  (b) train from scratch

In [ ]:
import time
from tensorflow import keras
es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
ytr_t = np.log1p(ytr) if LOG_TARGET else ytr
inv = (lambda p: np.expm1(p)) if LOG_TARGET else (lambda p: p)

# (a) transfer: load ACC model, freeze the first hidden layer, fine-tune the rest
base = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{BEST_X}.keras")
base.layers[0].trainable = False
base.compile(optimizer=keras.optimizers.Adam(3e-4), loss="mse", metrics=["mae"])
t = time.time(); base.fit(Xtr, ytr_t, validation_split=0.2, epochs=60, batch_size=256,
                          callbacks=[es], verbose=0); ft_dur = time.time()-t
ft_m = evaluate(yte, inv(base.predict(Xte, verbose=0).ravel())); ft_m.update(setting="fine-tuned (TL)", train_s=round(ft_dur,1))

# (b) scratch: identical architecture, fresh random weights, same limited data
scratch = keras.models.clone_model(base)        # clone_model re-initialises weights
for l in scratch.layers: l.trainable = True
scratch.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
t = time.time(); scratch.fit(Xtr, ytr_t, validation_split=0.2, epochs=60, batch_size=256,
                             callbacks=[es], verbose=0); sc_dur = time.time()-t
sc_m = evaluate(yte, inv(scratch.predict(Xte, verbose=0).ravel())); sc_m.update(setting="from scratch", train_s=round(sc_dur,1))

import pandas as pd
tl = pd.DataFrame([ft_m, sc_m]); tl.to_csv(f"{RESULTS_DIR}/transfer_learning.csv", index=False); tl

## Compare

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(13,4))
for i, metric in enumerate(["MSE","MAE","R2"]):
    ax[i].bar(tl.setting, tl[metric], color=["#2a9d8f","#e76f51"]); ax[i].set_title(metric)
    ax[i].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/05_transfer_learning.png", dpi=120); plt.show()
print("With a limited Salt&Tar train set, transfer learning is expected to match or beat training from",
      "scratch, especially on MSE/R2 and with shorter effective training.")